In [13]:
import numpy as np
import seaborn as sns
import pandas as pd

In [14]:
df = pd.read_csv('drug.csv')

In [15]:
df.head()

,name,substrates,side_effects,uses,Description,Action Class,Chemical Class,Therapeutic Class
0,augmentin 625 duo tablet,Penciclav 500 mg/125 mg Tablet Moxikind-CV 625...,Vomiting Nausea Diarrhea,Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,none,none,ANTI INFECTIVES
1,azithral 500 tablet,Zithrocare 500mg Tablet Azax 500 Tablet Zady 5...,Vomiting Nausea Abdominal pain Diarrhea,Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,Macrolides,Macrolides,ANTI INFECTIVES
2,ascoril ls syrup,Solvin LS Syrup Ambrodil-LX Syrup Zerotuss XP ...,Nausea Vomiting Diarrhea Upset stomach Stomach...,Treatment of Cough with mucus,Uses: Treatment of Cough with mucus. Side effe...,none,none,RESPIRATORY
3,allegra 120mg tablet,Lcfex Tablet Etofex 120mg Tablet Nexofex 120mg...,Headache Drowsiness Dizziness Nausea,Treatment of Sneezing and runny nose due to al...,Uses: Treatment of Sneezing and runny nose due...,H1 Antihistaminics (second Generation),Diphenylmethane Derivative,RESPIRATORY
4,avil 25 tablet,Eralet 25mg Tablet,Sleepiness Dryness in mouth,Treatment of Allergic conditions,Uses: Treatment of Allergic conditions. Side e...,H1 Antihistaminics (First Generation),Pyridines Derivatives,RESPIRATORY


In [16]:
df.shape

(224014, 8)

In [17]:
df.tail()

,name,substrates,side_effects,uses,Description,Action Class,Chemical Class,Therapeutic Class
224009,ziyapod 100mg oral suspension,Qc Pod Dry Syrup Actipod 50 Dry Syrup Orange-L...,Rash Nausea Diarrhea,Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,Cephalosporins: 3 generation,Broad Spectrum (Third & fourth generation ceph...,ANTI INFECTIVES
224010,zemhart 30mg tablet,Dilzex 30mg Tablet Diltiaz 30mg Tablet Triazem...,Headache Constipation Dizziness Fatigue Flushi...,Treatment of Hypertension (high blood pressure...,Uses: Treatment of Hypertension (high blood pr...,Calcium channel blockers- Nondihydropyridines,Benzothiazepine derivative,CARDIAC
224011,zivex 25mg tablet,HD Zine 25mg Tablet Hydrocas 25mg Tablet Hyzox...,Sedation Nausea Vomiting Upset stomach Constip...,Treatment of Anxiety Treatment of Skin conditi...,Uses: Treatment of Anxiety Treatment of Skin c...,H1 Antihistaminics (First Generation),Piperazine Derivative,RESPIRATORY
224012,zi fast 500mg injection,Zycin 500mg Injection Aziwok 500mg Injection A...,"Injection site reactions (pain, swelling, redn...",Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,Macrolides,Macrolides,ANTI INFECTIVES
224013,zyvocol 1% dusting powder,Canazole Dusting Powder Clotrex Dusting Powder...,Blisters Skin peeling Swelling Application sit...,Treatment of Fungal skin infections,Uses: Treatment of Fungal skin infections. Sid...,Fungal ergosterol synthesis inhibitor,Azole derivatives {Imidazoles},DERMA


In [18]:
""" 
PIPELINE OVERVIEW:
  Step 1  → Data Loading & Exploration
  Step 2  → Target Engineering & Class Selection
  Step 3  → Input Text Construction (feature engineering for NLP)
  Step 4  → Label Encoding (text labels → integers)
  Step 5  → Train/Validation/Test Split
  Step 6  → Tokenization (text → token IDs) — THE VECTORIZATION STEP
  Step 7  → PyTorch Dataset & DataLoader (batching machinery)
  Step 8  → Load Pretrained BERT Model (Transfer Learning)
  Step 9  → Training Loop (fine-tuning)
  Step 10 → Evaluation (metrics & analysis)
""" 

' \nPIPELINE OVERVIEW:\n  Step 1  → Data Loading & Exploration\n  Step 2  → Target Engineering & Class Selection\n  Step 3  → Input Text Construction (feature engineering for NLP)\n  Step 4  → Label Encoding (text labels → integers)\n  Step 5  → Train/Validation/Test Split\n  Step 6  → Tokenization (text → token IDs) — THE VECTORIZATION STEP\n  Step 7  → PyTorch Dataset & DataLoader (batching machinery)\n  Step 8  → Load Pretrained BERT Model (Transfer Learning)\n  Step 9  → Training Loop (fine-tuning)\n  Step 10 → Evaluation (metrics & analysis)\n'

In [19]:
import sys
print(sys.executable)

/Library/Frameworks/Python.framework/Versions/3.13/bin/python3.13


In [20]:
# !pip install torch torchvision torchaudio

In [21]:
# pip install --upgrade pip

In [22]:
#!pip install transformers

In [23]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import warnings, time
 
warnings.filterwarnings("ignore")
 
# Detect GPU / CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n🖥️  Using device: {DEVICE}")


🖥️  Using device: cpu


In [24]:
print('='*70)
print(' Step 1 : Data Loading and Exploring ') 
print('='*70)

 Step 1 : Data Loading and Exploring 


In [25]:
print(f"\n Dataset shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)")
print("\n Column names:")
for col in df.columns:
    print(f"   • {col}")


 Dataset shape: (224014, 8)  (224,014 rows × 8 columns)

 Column names:
   • name
   • substrates
   • side_effects
   • uses
   • Description
   • Action Class
   • Chemical Class
   • Therapeutic Class


In [26]:
print("\n Unique values per column:")
for col in df.columns:
    print(f"   {col}: {df[col].nunique()} unique")


 Unique values per column:
   name: 222825 unique
   substrates: 43296 unique
   side_effects: 3099 unique
   uses: 1078 unique
   Description: 3650 unique
   Action Class: 432 unique
   Chemical Class: 873 unique
   Therapeutic Class: 23 unique


In [27]:
print("\n Top 5 Therapeutic Classes ")
print(df['Therapeutic Class'].value_counts().head(5))


 Top 5 Therapeutic Classes 
Therapeutic Class
ANTI INFECTIVES      50459
GASTRO INTESTINAL    30442
PAIN ANALGESICS      29187
NEURO CNS            21524
RESPIRATORY          21284
Name: count, dtype: int64


In [28]:
df['name'].duplicated().sum()

np.int64(1189)

In [29]:
df['name'] = df['name'].str.strip().str.lower()

In [30]:
df['name'].duplicated().sum()

np.int64(1189)

In [31]:
df[df.duplicated(subset=['name'], keep=False)].sort_values(by='name')

,name,substrates,side_effects,uses,Description,Action Class,Chemical Class,Therapeutic Class
6419,a rex drop,NaN,Sedation Nausea Vomiting Upset stomach Constip...,Treatment of Anxiety Treatment of Skin conditi...,Uses: Treatment of Anxiety Treatment of Skin c...,H1 Antihistaminics (First Generation),Piperazine Derivative,RESPIRATORY
12408,a rex drop,Azon 6mg Drop Pru 6mg Drop Hydrox 6mg Drop Ktr...,Sedation Nausea Vomiting Upset stomach Constip...,Treatment of Anxiety Treatment of Skin conditi...,Uses: Treatment of Anxiety Treatment of Skin c...,H1 Antihistaminics (First Generation),Piperazine Derivative,RESPIRATORY
3949,abatitor 250mg tablet,Zelgor 250mg Tablet Abitate 250mg Tablet Xbira...,Edema (swelling) Vomiting Decreased potassium ...,Prostate cancer,Uses: Prostate cancer. Side effects: Edema (sw...,Anticancer-others,Androstene Derivative,ANTI NEOPLASTICS
15910,abatitor 250mg tablet,Zelgor 250mg Tablet Abitate 250mg Tablet Xbira...,Edema (swelling) Vomiting Decreased potassium ...,Prostate cancer,Uses: Prostate cancer. Side effects: Edema (sw...,Anticancer-others,Androstene Derivative,ANTI NEOPLASTICS
25215,abicold tablet,MH Cold Tablet Nicip Cold & Flu Tablet Sneezy ...,Nausea Diarrhea Vomiting Dryness in mouth Inso...,Treatment of Common cold,Uses: Treatment of Common cold. Side effects: ...,none,none,RESPIRATORY
...,...,...,...,...,...,...,...,...
221726,zyser 10mg tablet,Serowel Tablet Serperite 10mg Tablet Stardase ...,No common side effects seen,Pain relief Swelling,Uses: Pain relief Swelling. Side effects: No c...,Proteolytic Enzymes,Proteolytic Enzyme,PAIN ANALGESICS
218155,zythro 500 tablet,Azax 500 Tablet Zady 500 Tablet Trulimax 500mg...,Vomiting Nausea Abdominal pain Diarrhea,Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,Macrolides,Macrolides,ANTI INFECTIVES
221317,zythro 500 tablet,Zithrocare 500mg Tablet Azax 500 Tablet Zady 5...,Vomiting Nausea Abdominal pain Diarrhea,Treatment of Bacterial infections,Uses: Treatment of Bacterial infections. Side ...,Macrolides,Macrolides,ANTI INFECTIVES
217577,zytus a syrup,Batrox AM Syrup Ambo Syrup Cofeon Syrup Yetbro...,Nausea Diarrhea Vomiting Dizziness Headache Ra...,Cough,Uses: Cough. Side effects: Nausea Diarrhea Vom...,none,none,RESPIRATORY


In [32]:
print(df.duplicated())

0         False
1         False
2         False
3         False
4         False
          ...  
224009    False
224010    False
224011    False
224012    False
224013    False
Length: 224014, dtype: bool


In [33]:
df['completeness'] = df.notna().sum(axis=1)
df = df.sort_values('completeness', ascending=False) \
       .drop_duplicates(subset=['name'], keep='first')
df = df.drop(columns=['completeness'])

In [34]:
print(f"Dataset shape: {df.shape}  ({df.shape[0]:,} rows × {df.shape[1]} columns)")

Dataset shape: (222825, 8)  (222,825 rows × 8 columns)


In [35]:
print("="*70)
print("STEP 2: Target Engineering")
print("="*70)

STEP 2: Target Engineering


In [36]:
class_counts = df['Action Class'].value_counts()
min_samples = 100
valid_classes = [
    c for c in class_counts.index
    if class_counts[c] >= min_samples and c.strip().lower() != 'none'
]
print(f"Classes included: {len(valid_classes)}")  

Classes included: 145


In [37]:
print(f"We keep top {len(valid_classes)} (excluding 'none') for clean multi-class classification")

We keep top 145 (excluding 'none') for clean multi-class classification


In [38]:
df_model = df[df['Action Class'].isin(valid_classes)].copy().reset_index(drop=True)

In [39]:
df_model

,name,substrates,side_effects,uses,Description,Action Class,Chemical Class,Therapeutic Class
0,naropin 0.75% injection,Ropee 0.75% Injection Ropin 0.75% Injection Tr...,Hypotension (low blood pressure) Nausea Vomiti...,Local anesthesia (Numb tissues in a specific a...,Uses: Local anesthesia (Numb tissues in a spec...,Local anaesthetic (Amides),Amide derivative,PAIN ANALGESICS
1,nurohist 24mg tablet,Histavert 24 Tablet Betway 24mg Tablet Nelvert...,Headache Indigestion Nausea Stomach pain Bloating,Treatment of Meniere's disease,Uses: Treatment of Meniere's disease. Side eff...,Histamine analog- Meniere's Disease,Histamine Analog,NEURO CNS
2,nando 50mg injection,Decanan 50mg Injection Anudec 50mg Injection D...,Edema (swelling) Nausea Breast enlargement Acne,Treatment of Post menopausal osteoporosis,Uses: Treatment of Post menopausal osteoporosi...,Anabolic steroid,Anabolic steroid,HORMONES
3,nucoshine 120mg tablet,Intacoxia 120 Tablet Coxnuro 120mg Tablet Eton...,Stomach pain Diarrhea,Pain relief,Uses: Pain relief. Side effects: Stomach pain ...,NSAID's -Selective COX-2 Inhibitors,Sulfone and Pyridine Derivative,PAIN ANALGESICS
4,neurocit 500 tablet,Citinova Tablet Strozina 500 Tablet Citimac Ta...,Decreased blood pressure Arrhythmia (irregular...,Treatment of Stroke Treatment of Head injury T...,Uses: Treatment of Stroke Treatment of Head in...,Nootropic agent,Pyrimidine Ribonucleoside Diphosphate,NEURO CNS
...,...,...,...,...,...,...,...,...
115777,eprozar 400mg tablet,NaN,Headache Dizziness Runny nose Nausea Diarrhea ...,Hypertension (high blood pressure) Prevention ...,Uses: Hypertension (high blood pressure) Preve...,Angiotensin receptor blockers(ARB),Benzoic acids derivative,CARDIAC
115778,prednol 0.3% eye ointment,NaN,Eye irritation Watery eyes Burning sensation,Redness and swelling in the eye,Uses: Redness and swelling in the eye. Side ef...,Glucocorticoids,Glucocorticoids,OPHTHAL
115779,factus 40mg tablet sr,NaN,Diarrhea Headache Increased liver enzymes Naus...,Treatment of Gout,Uses: Treatment of Gout. Side effects: Diarrhe...,Xanthine oxidase Inhibitors-gout,Thiazole Derivative,PAIN ANALGESICS
115780,carvitop 6.25mg tablet ir,NaN,Decreased blood pressure Headache Fatigue Dizz...,Treatment of Hypertension (high blood pressure...,Uses: Treatment of Hypertension (high blood pr...,Alpha & beta blocker,Carbazole & propanol derivative,CARDIAC


In [40]:
print(f"📊 Class distribution in filtered dataset:")
for cls, cnt in df_model['Action Class'].value_counts().items():
    bar = "█" * (cnt // 400)
    print(f"   {cls:<45} {cnt:>6}  {bar}")

📊 Class distribution in filtered dataset:
   Cephalosporins: 3 generation                   10857  ███████████████████████████
   Quinolones/ Fluroquinolones                     6966  █████████████████
   Fungal ergosterol synthesis inhibitor           5908  ██████████████
   Proton pump inhibitors                          5714  ██████████████
   Macrolides                                      5006  ████████████
   Glucocorticoids                                 4753  ███████████
   H1 Antihistaminics (second Generation)          3677  █████████
   HMG CoA inhibitors (statins)                    2538  ██████
   Benzodiazepines                                 2443  ██████
   Cephalosporins: 2nd generation                  2439  ██████
   NSAID's- Non-Selective COX 1&2 Inhibitors (acetic acid)   2419  ██████
   Serotonin antagonists (5-HT3 antagonists)       2367  █████
   Atypical Antipsychotics                         2358  █████
   Aminoglycosides                                 1946 

In [41]:
print("="*70)
print("STEP 3: Input Text Generation")
print("="*70)

STEP 3: Input Text Generation


In [42]:
df_model['input text'] = (
    "Drug: " + df_model['name'].fillna('unknown') + ". " +
    "Uses: " + df_model['uses'].fillna('') + ". "
    + "Side Effects: " + df_model['side_effects'].fillna('')
)

In [43]:
print("="*70)
print("STEP 4: Label Encoding")
print("="*70)

STEP 4: Label Encoding


In [44]:
le = LabelEncoder()
df_model['label'] = le.fit_transform(df_model['Action Class'])
 
print(f"\n🔢 Label mapping (class name → integer ID):")
for cls, idx in zip(le.classes_, range(len(le.classes_))):
    print(f"   {idx} → {cls}")
 
NUM_CLASSES = len(le.classes_)
print(f"\n   Total classes: {NUM_CLASSES}")


🔢 Label mapping (class name → integer ID):
   0 → 5 alpha- reductase inhibitor (5ARI)
   1 → 5-Nitroimidazole (Antiprotozoal & Antibacterial)
   2 → Acetylcysteine -Mucolytic
   3 → Alkaloids-cytotoxic agents
   4 → Alkylating agent
   5 → Alpha & beta blocker
   6 → Alpha 2 delta ligands (AED)
   7 → Alpha-glucosidase inhibitors
   8 → Amino acids
   9 → Aminoglycosides
   10 → Anabolic steroid
   11 → Analgesic & Antipyretic-PCM
   12 → Angiotensin receptor blockers(ARB)
   13 → Angiotensin-converting enzyme (ACE) inhibitors
   14 → Antacids
   15 → Anti-Ulcerants
   16 → Anticancer-others
   17 → Anticholinergic- centrally acting
   18 → Anticholinergics
   19 → Antifibrinolytic
   20 → Antifungal Others
   21 → Antimalarial- Aminoquinolines
   22 → Antimalarial- Artemisinin and derivatives
   23 → Antimalarial- others
   24 → Antimetabolite- Methotrexate
   25 → Antimetabolites
   26 → Antimicrotubule agents- Taxanes
   27 → Antiprotozoal agents
   28 → Antiseptics and disinfectan

In [45]:
print("="*70)
print("STEP 5: Train / Test / Split ")
print("="*70)

STEP 5: Train / Test / Split 


In [46]:
MAX_SAMPLES = 50000
df_sample = df_model.groupby('Action Class', group_keys=False).apply(
    lambda x: x.sample(min(len(x), MAX_SAMPLES // NUM_CLASSES), random_state=42)
).reset_index(drop=True)

In [47]:
df_sample

,name,substrates,side_effects,uses,Description,Action Class,Chemical Class,Therapeutic Class,input text,label
0,dutabest 0.5mg tablet,Dutastex 0.5mg Tablet DavaIndia Dutasteride 0....,Decreased sperm count Impotence Decreased libi...,Benign prostatic hyperplasia,Uses: Benign prostatic hyperplasia. Side effec...,5 alpha- reductase inhibitor (5ARI),Androgen Derivative,UROLOGY,Drug: dutabest 0.5mg tablet. Uses: Benign pros...,0
1,finadex tablet,Growvita 1 Tablet Fintross 1mg Tablet Finacet ...,Decreased libido Erectile dysfunction Ejaculat...,Treatment of Hair loss,Uses: Treatment of Hair loss. Side effects: De...,5 alpha- reductase inhibitor (5ARI),Androgen Derivative,UROLOGY,Drug: finadex tablet. Uses: Treatment of Hair ...,0
2,dutastic 0.5mg tablet,Dutastex 0.5mg Tablet DavaIndia Dutasteride 0....,Decreased sperm count Impotence Decreased libi...,Benign prostatic hyperplasia,Uses: Benign prostatic hyperplasia. Side effec...,5 alpha- reductase inhibitor (5ARI),Androgen Derivative,UROLOGY,Drug: dutastic 0.5mg tablet. Uses: Benign pros...,0
3,astefin 1mg tablet,Growvita 1 Tablet Fintross 1mg Tablet Finacet ...,Decreased libido Erectile dysfunction Ejaculat...,Treatment of Hair loss,Uses: Treatment of Hair loss. Side effects: De...,5 alpha- reductase inhibitor (5ARI),Androgen Derivative,UROLOGY,Drug: astefin 1mg tablet. Uses: Treatment of H...,0
4,fincept 5mg tablet,Finar 5mg Tablet Routwin 5mg Tablet Finastat 5...,Decreased libido Erectile dysfunction Ejaculat...,Treatment of Hair loss,Uses: Treatment of Hair loss. Side effects: De...,5 alpha- reductase inhibitor (5ARI),Androgen Derivative,UROLOGY,Drug: fincept 5mg tablet. Uses: Treatment of H...,0
...,...,...,...,...,...,...,...,...,...,...
37981,vorisung 50mg tablet,Alenfast 50mg Tablet Cyndol 50mg Tablet Tapcyn...,Nausea Dizziness Vomiting Sleepiness,Moderate to severe pain,Uses: Moderate to severe pain. Side effects: N...,μ-opioid receptor & norepinephrine reuptake in...,Phenylpropanes Derivative,PAIN ANALGESICS,Drug: vorisung 50mg tablet. Uses: Moderate to ...,144
37982,iontep 100mg tablet,Niap 100mg Tablet Vorth TP 100mg Tablet Alenfa...,Nausea Dizziness Vomiting Sleepiness,Moderate to severe pain,Uses: Moderate to severe pain. Side effects: N...,μ-opioid receptor & norepinephrine reuptake in...,Phenylpropanes Derivative,PAIN ANALGESICS,Drug: iontep 100mg tablet. Uses: Moderate to s...,144
37983,orthenta 100mg tablet,Niap 100mg Tablet Vorth TP 100mg Tablet Alenfa...,Nausea Dizziness Vomiting Sleepiness,Moderate to severe pain,Uses: Moderate to severe pain. Side effects: N...,μ-opioid receptor & norepinephrine reuptake in...,Phenylpropanes Derivative,PAIN ANALGESICS,Drug: orthenta 100mg tablet. Uses: Moderate to...,144
37984,cyndol 75mg tablet,Vorth TP 75mg Tablet Dol Proxyvon 75mg Tablet ...,Nausea Dizziness Vomiting Sleepiness,Moderate to severe pain,Uses: Moderate to severe pain. Side effects: N...,μ-opioid receptor & norepinephrine reuptake in...,Phenylpropanes Derivative,PAIN ANALGESICS,Drug: cyndol 75mg tablet. Uses: Moderate to se...,144


In [48]:
texts  = df_sample['input text'].tolist()
labels = df_sample['label'].tolist()

In [49]:
 # First split: train vs. temp (val + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

In [50]:
# Second split: val vs. test (each gets half of temp → 15% each)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

In [51]:
print(f"\nSplit sizes:")
print(f"   Training   : {len(X_train):>5} samples (70%)")
print(f"   Validation : {len(X_val):>5} samples (15%)")
print(f"   Test       : {len(X_test):>5} samples (15%)")


Split sizes:
   Training   : 26590 samples (70%)
   Validation :  5698 samples (15%)
   Test       :  5698 samples (15%)


In [52]:
print("="*70)
print("STEP 6: Tokenization ")
print("="*70)

STEP 6: Tokenization 


In [336]:
MODEL_NAME = "dmis-lab/biobert-base-cased-v1.2"
MAX_LEN = 128
 
print(f"\n Loading tokenizer: '{MODEL_NAME}'")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded. Vocabulary size: {tokenizer.vocab_size:,} tokens")


 Loading tokenizer: 'dmis-lab/biobert-base-cased-v1.2'


config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Tokenizer loaded. Vocabulary size: 28,996 tokens


In [337]:
print(f" Tokenize all {len(X_train) + len(X_test) + len(X_val)} texts ... ")

 Tokenize all 37986 texts ... 


In [338]:
t0 = time.time()

In [339]:
def tokenize_texts(texts_list):
    return tokenizer(
        texts_list,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
train_encodings = tokenize_texts(X_train)
val_encodings = tokenize_texts(X_val)
test_encodings = tokenize_texts(X_test)

print(f"Done in {(time.time()-t0):.1f}s")
print(f"Shape of train input_ids: {train_encodings['input_ids'].shape}")

Done in 4.8s
Shape of train input_ids: torch.Size([26590, 128])


In [340]:
print("="*70)
print("STEP 7: PyTorch Dataset & Dataloader ")
print("="*70)

STEP 7: PyTorch Dataset & Dataloader 


In [341]:
class DrugDataset(Dataset):
    
    def __init__(self, encodings, labels):
        self.input_ids      = encodings['input_ids']
        self.attention_mask = encodings['attention_mask']
        self.labels         = torch.tensor(labels, dtype=torch.long)
 
    def __len__(self):
        return len(self.labels)
 
    def __getitem__(self, idx):
        return {
            'input_ids':      self.input_ids[idx],
            'attention_mask': self.attention_mask[idx],
            'label':          self.labels[idx]
        }
 
 
BATCH_SIZE = 32
 
train_dataset = DrugDataset(train_encodings, y_train)
val_dataset = DrugDataset(val_encodings,   y_val)
test_dataset = DrugDataset(test_encodings,  y_test)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)
 
print(f"\n Datasets and loaders created:")
print(f" Train: {len(train_dataset):,} samples - {len(train_loader)} batches of {BATCH_SIZE}")
print(f" Val: {len(val_dataset):,} samples - {len(val_loader)} batches")
print(f" Test: {len(test_dataset):,} samples - {len(test_loader)} batches")
 
# Show one batch
sample_batch = next(iter(train_loader))
print(f"One batch shape:")
print(f"input_ids: {sample_batch['input_ids'].shape}  (batch × max_len)")
print(f"attention_mask: {sample_batch['attention_mask'].shape}")
print(f"labels: {sample_batch['label'].shape}  (batch,)")


 Datasets and loaders created:
 Train: 26,590 samples - 831 batches of 32
 Val: 5,698 samples - 179 batches
 Test: 5,698 samples - 179 batches
One batch shape:
input_ids: torch.Size([32, 128])  (batch × max_len)
attention_mask: torch.Size([32, 128])
labels: torch.Size([32])  (batch,)


In [342]:
print("="*70)
print("STEP 8: Load PreTrained BERT ")
print("="*70)

STEP 8: Load PreTrained BERT 


In [343]:
print(f" Loading '{MODEL_NAME}' with {NUM_CLASSES}-class classification head...")
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_CLASSES
)
model = model.to(DEVICE)

 Loading 'dmis-lab/biobert-base-cased-v1.2' with 145-class classification head...


[transformers] You passed `num_labels=145` which is incompatible to the `id2label` map of length `2`.


pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those

model.safetensors:   0%|          | 0.00/436M [00:00<?, ?B/s]

In [344]:
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

In [345]:
print(f"Model loaded and moved to {DEVICE}")
print(f"Total parameters   : {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Model loaded and moved to cpu
Total parameters   : 108,421,777
Trainable parameters: 108,421,777


In [346]:
print("="*70)
print("STEP 9: Training Loop ")
print("="*70)

STEP 9: Training Loop 


In [347]:
EPOCHS = 3
LR = 2e-5
WARMUP_PCT = 0.10

In [348]:
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_PCT)

In [350]:
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
loss_fn = nn.CrossEntropyLoss()

In [351]:
# Training Function 

In [352]:
def train_one_epoch(model, loader, optimizer, scheduler, loss_fn, device, epoch):
    model.train()  
    total_loss, correct, total = 0.0, 0, 0
 
    for step, batch in enumerate(loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
 
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits         
 
        loss = loss_fn(logits, labels)   
        total_loss += loss.item()
 
        loss.backward()                  
 
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
 
        # Weight Update
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()             
 
        preds = torch.argmax(logits, dim=1)  
        correct += (preds == labels).sum().item()
        total += labels.size(0)
 
        
        if (step + 1) % 40 == 0 or (step + 1) == len(loader):
            print(f"Epoch {epoch} | Step {step+1}/{len(loader)} | "
                  f"Loss: {total_loss/(step+1):.4f} | "
                  f"Acc: {correct/total:.4f}")
 
    return total_loss / len(loader), correct / total

In [353]:
def evaluate(model, loader, loss_fn, device):
    model.eval() 
    total_loss, all_preds, all_labels = 0.0, [], []
 
    with torch.no_grad():  
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
 
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits  = outputs.logits
            loss = loss_fn(logits, labels)
            total_loss += loss.item()
 
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    accuracy = accuracy_score(all_labels, all_preds)
    return avg_loss, accuracy, all_preds, all_labels

In [355]:
print(f"Starting training for {EPOCHS} epochs \n")
best_val_acc = 0.0
training_history = []
 
for epoch in range(1, EPOCHS + 1):
    print(f"\n{'─'*60}")
    print(f"EPOCH {epoch}/{EPOCHS}")
    print(f"{'─'*60}")
 
    epoch_start = time.time()
 
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler, loss_fn, DEVICE, epoch
    )
 
    val_loss, val_acc, _, _ = evaluate(model, val_loader, loss_fn, DEVICE)
 
    elapsed = time.time() - epoch_start
 
    print(f" Epoch {epoch} Summary:")
    print(f" Train Loss: {train_loss:.4f}  |  Train Acc: {train_acc:.4f}")
    print(f" Val Loss: {val_loss:.4f}  |  Val Acc: {val_acc:.4f}")
    print(f" Time: {elapsed:.1f}s")
 
    training_history.append({
        'epoch': epoch,
        'train_loss': train_loss, 'train_acc': train_acc,
        'val_loss': val_loss,   'val_acc': val_acc
    })
 
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        model.save_pretrained("./drug_model/")
        tokenizer.save_pretrained("./drug_model/")
        print(f" New best model saved (val_acc={val_acc:.4f})")

Starting training for 3 epochs 


────────────────────────────────────────────────────────────
EPOCH 1/3
────────────────────────────────────────────────────────────
Epoch 1 | Step 40/831 | Loss: 5.0184 | Acc: 0.0063
Epoch 1 | Step 80/831 | Loss: 4.9652 | Acc: 0.0102
Epoch 1 | Step 120/831 | Loss: 4.9075 | Acc: 0.0169
Epoch 1 | Step 160/831 | Loss: 4.8317 | Acc: 0.0359
Epoch 1 | Step 200/831 | Loss: 4.7250 | Acc: 0.0841
Epoch 1 | Step 240/831 | Loss: 4.5973 | Acc: 0.1480
Epoch 1 | Step 280/831 | Loss: 4.4585 | Acc: 0.2136
Epoch 1 | Step 320/831 | Loss: 4.3085 | Acc: 0.2783
Epoch 1 | Step 360/831 | Loss: 4.1635 | Acc: 0.3335
Epoch 1 | Step 400/831 | Loss: 4.0170 | Acc: 0.3852
Epoch 1 | Step 440/831 | Loss: 3.8749 | Acc: 0.4285
Epoch 1 | Step 480/831 | Loss: 3.7393 | Acc: 0.4644
Epoch 1 | Step 520/831 | Loss: 3.6037 | Acc: 0.4980
Epoch 1 | Step 560/831 | Loss: 3.4765 | Acc: 0.5270
Epoch 1 | Step 600/831 | Loss: 3.3545 | Acc: 0.5528
Epoch 1 | Step 640/831 | Loss: 3.2413 | Acc: 0.5760
Epoc

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 New best model saved (val_acc=0.9768)

────────────────────────────────────────────────────────────
EPOCH 2/3
────────────────────────────────────────────────────────────
Epoch 2 | Step 40/831 | Loss: 0.8216 | Acc: 0.9719
Epoch 2 | Step 80/831 | Loss: 0.7857 | Acc: 0.9727
Epoch 2 | Step 120/831 | Loss: 0.7464 | Acc: 0.9740
Epoch 2 | Step 160/831 | Loss: 0.7182 | Acc: 0.9738
Epoch 2 | Step 200/831 | Loss: 0.6823 | Acc: 0.9750
Epoch 2 | Step 240/831 | Loss: 0.6552 | Acc: 0.9755
Epoch 2 | Step 280/831 | Loss: 0.6326 | Acc: 0.9752
Epoch 2 | Step 320/831 | Loss: 0.6070 | Acc: 0.9759
Epoch 2 | Step 360/831 | Loss: 0.5856 | Acc: 0.9763
Epoch 2 | Step 400/831 | Loss: 0.5630 | Acc: 0.9773
Epoch 2 | Step 440/831 | Loss: 0.5441 | Acc: 0.9778
Epoch 2 | Step 480/831 | Loss: 0.5257 | Acc: 0.9783
Epoch 2 | Step 520/831 | Loss: 0.5090 | Acc: 0.9788
Epoch 2 | Step 560/831 | Loss: 0.4926 | Acc: 0.9792
Epoch 2 | Step 600/831 | Loss: 0.4776 | Acc: 0.9798
Epoch 2 | Step 640/831 | Loss: 0.4641 | Acc: 0.980

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 New best model saved (val_acc=0.9921)

────────────────────────────────────────────────────────────
EPOCH 3/3
────────────────────────────────────────────────────────────
Epoch 3 | Step 40/831 | Loss: 0.1612 | Acc: 0.9977
Epoch 3 | Step 80/831 | Loss: 0.1696 | Acc: 0.9918
Epoch 3 | Step 120/831 | Loss: 0.1646 | Acc: 0.9911
Epoch 3 | Step 160/831 | Loss: 0.1604 | Acc: 0.9916
Epoch 3 | Step 200/831 | Loss: 0.1544 | Acc: 0.9927
Epoch 3 | Step 240/831 | Loss: 0.1510 | Acc: 0.9923
Epoch 3 | Step 280/831 | Loss: 0.1483 | Acc: 0.9927
Epoch 3 | Step 320/831 | Loss: 0.1450 | Acc: 0.9930
Epoch 3 | Step 360/831 | Loss: 0.1439 | Acc: 0.9927
Epoch 3 | Step 400/831 | Loss: 0.1407 | Acc: 0.9929
Epoch 3 | Step 440/831 | Loss: 0.1385 | Acc: 0.9930
Epoch 3 | Step 480/831 | Loss: 0.1373 | Acc: 0.9926
Epoch 3 | Step 520/831 | Loss: 0.1359 | Acc: 0.9924
Epoch 3 | Step 560/831 | Loss: 0.1349 | Acc: 0.9923
Epoch 3 | Step 600/831 | Loss: 0.1329 | Acc: 0.9923
Epoch 3 | Step 640/831 | Loss: 0.1314 | Acc: 0.992

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

 New best model saved (val_acc=0.9933)


In [356]:
"""

 model = AutoModelForSequenceClassification.from_pretrained("./drug_model/")
 tokenizer = AutoTokenizer.from_pretrained("./drug_model/")

"""

'\n\n model = AutoModelForSequenceClassification.from_pretrained("./drug_model/")\n tokenizer = AutoTokenizer.from_pretrained("./drug_model/")\n\n'

In [357]:
print("="*70)
print("STEP 10: Final Test Evaluation ")
print("="*70)

STEP 10: Final Test Evaluation 


In [358]:
print(f"Loading best model (valuation accuracy = {best_val_acc:.4f}) or {best_val_acc * 100:.2f}% ")

Loading best model (valuation accuracy = 0.9933) or 99.33% 


In [359]:
model = AutoModelForSequenceClassification.from_pretrained("./drug_model/")
tokenizer = AutoTokenizer.from_pretrained("./drug_model/")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [360]:
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, loss_fn, DEVICE) 

print(f" Test Results : ")
print(f" Test Loss : {test_loss:.4f}")
print(f" Test Accuracy : {test_acc:.4f}  ({test_acc*100:.1f}%)")


 Test Results : 
 Test Loss : 0.0793
 Test Accuracy : 0.9916  (99.2%)


In [361]:
# Training History
print(f"Training History:")
print(f"{'Epoch':>5}  {'Train Loss':>10}  {'Train Acc':>9}  {'Val Loss':>8}  {'Val Acc':>7}")
print("─" * 70)
for h in training_history:
    print(f"{h['epoch']:>5}  {h['train_loss']:>10.4f}  {h['train_acc']:>9.4f}  "
          f"{h['val_loss']:>8.4f}  {h['val_acc']:>7.4f}")

Training History:
Epoch  Train Loss  Train Acc  Val Loss  Val Acc
──────────────────────────────────────────────────────────────────────
    1      2.7552     0.6616    0.6631   0.9768
    2      0.4043     0.9824    0.1170   0.9921
    3      0.1255     0.9927    0.0726   0.9933


In [362]:
print("="*70)
print("FINAL STEP : Inference on Raw Text ")
print("="*70)

FINAL STEP : Inference on Raw Text 


In [371]:
def predict_action_class(text, model, tokenizer, label_encoder, device, max_len=128):
    
    model.eval()
    encoding = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
 
    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)

        logits  = outputs.logits                          
        probs   = torch.softmax(logits, dim=1)[0]          
        pred_id = torch.argmax(probs).item()
 
    pred_class = label_encoder.inverse_transform([pred_id])[0]
 
    # Top 3 predictions with probabilities
    top3 = torch.topk(probs, k=3)
    top3_classes = label_encoder.inverse_transform(top3.indices.cpu().numpy())
    top3_probs   = top3.values.cpu().numpy()
 
    return pred_class, list(zip(top3_classes, top3_probs))
 

In [372]:
# Example 

In [383]:
text1 = ("Drug: rosuvastatin 20mg tablet. "
         "Uses: Management of high cholesterol and reduction of cardiovascular disease risk. "
         "Side Effects: Muscle pain, headache, weakness, and abdominal pain.")

text2 = ("Drug: montelukast 10mg tablet. "
         "Uses: Prevention and long-term treatment of asthma and relief of allergic rhinitis symptoms. "
         "Side Effects: Headache, stomach ache, mood changes, and vivid dreams.")

text3 = ("Drug: escitalopram 10mg tablet. "
         "Uses: Treatment of major depressive disorder and generalized anxiety disorder. "
         "Side Effects: Nausea, dry mouth, sweating, and fatigue.")

In [384]:
for i, text in enumerate([text1, text2, text3], 1):
    pred, top3 = predict_action_class(text, model, tokenizer, le, DEVICE)
    print(f"Example {i}:")
    print(f"Input  : {text[:100]}...")
    print(f"Predicted Action Class: {pred}")
    print(f"Top-3 predictions:")
    for cls, prob in top3:
        bar = "█" * int(prob * 30)
        print(f"{prob:.3f} {bar} {cls}")

Example 1:
Input  : Drug: rosuvastatin 20mg tablet. Uses: Management of high cholesterol and reduction of cardiovascular...
Predicted Action Class: HMG CoA inhibitors (statins)
Top-3 predictions:
0.743 ██████████████████████ HMG CoA inhibitors (statins)
0.014  Amino acids
0.008  Immunosuppressant- Calcineurin inhibitors
Example 2:
Input  : Drug: montelukast 10mg tablet. Uses: Prevention and long-term treatment of asthma and relief of alle...
Predicted Action Class: H1 Antihistaminics (second Generation)
Top-3 predictions:
0.573 █████████████████ H1 Antihistaminics (second Generation)
0.077 ██ Leukotriene antagonists
0.054 █ Glucocorticoids
Example 3:
Input  : Drug: escitalopram 10mg tablet. Uses: Treatment of major depressive disorder and generalized anxiety...
Predicted Action Class: Benzodiazepines
Top-3 predictions:
0.627 ██████████████████ Benzodiazepines
0.081 ██ Selective Seretonin Reuptake inhibitors (SSRIs)
0.029  Serotonin-norepinephrine reuptake inhibitors (SNRIs)


In [379]:
# Check if Model predicts same result for difference comprehension

In [380]:
texta = ("Metformin 850mg tablets are used to manage type 2 diabetes and improve blood sugar control, but they can cause side effects like nausea, an upset stomach, diarrhea, and a metallic taste in the mouth.")

textb = ("People take a metformin 850mg tablet to help control their blood sugar and manage type 2 diabetes, though it can sometimes cause nausea, stomach upset, diarrhea, or a weird metallic taste.")

textc = ("I take metformin 850mg to manage my type 2 diabetes and keep my blood sugar under control, but the side effects can include an upset stomach, nausea, diarrhea, and a metallic taste in my mouth.")

In [382]:
for i, text in enumerate([texta, textb, textc], 1):
    pred, top3 = predict_action_class(text, model, tokenizer, le, DEVICE)
    print(f"Example {i}:")
    print(f"Input  : {text[:100]}...")
    print(f"Predicted Action Class: {pred}")
    print("\n")

Example 1:
Input  : Metformin 850mg tablets are used to manage type 2 diabetes and improve blood sugar control, but they...
Predicted Action Class: DPP-4 inhibitors


Example 2:
Input  : People take a metformin 850mg tablet to help control their blood sugar and manage type 2 diabetes, t...
Predicted Action Class: DPP-4 inhibitors


Example 3:
Input  : I take metformin 850mg to manage my type 2 diabetes and keep my blood sugar under control, but the s...
Predicted Action Class: DPP-4 inhibitors




In [385]:
#Show all 3 Predictions 

In [53]:
import pickle
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)
print("Label encoder saved ... ")

Label encoder saved ... 
